In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Conv1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# =========================
# Configuration and constants
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100          # time-normalized points per cycle (101 for exact 0-100%)
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']
FEATURE_COLUMNS = [
    'IMU101_v0', 'IMU101_v1',
    'IMU103_v0', 'IMU103_v1',
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1',
    #'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6',
    #'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]

# Model hyperparameters
UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32

# Checkpoint naming (kept distinct from the old padded-input runs)
WEIGHTS_PREFIX = 'best_timenorm'


def resample_cycle(features, target_len=TARGET_LEN):
    """Time-normalize one cycle onto a fixed number of points (0-100% of cycle)."""
    n = features.shape[0]
    kind = 'cubic' if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_and_split_data():
    """Load cycles, select features, time-normalize, encode labels, and split."""
    raw = []

    if not os.path.exists(ROOT_DIR):
        return None, None, None, None, 0, 0, None

    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                file_path = os.path.join(root, filename)

                # Parse label and cycle id from filename
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cycle_id_str = raw_id_part.split('_')[-1]

                if label not in VALID_LABELS or not cycle_id_str.isdigit():
                    continue

                try:
                    df = pd.read_csv(file_path)
                except Exception:
                    continue

                # Feature selection
                if any(col not in df.columns for col in FEATURE_COLUMNS):
                    continue
                df = df[FEATURE_COLUMNS]
                cycle_features = df.values
                if cycle_features.shape[0] < 2 or cycle_features.shape[1] == 0:
                    continue

                raw.append((cycle_features, label, file_path))

    if not raw:
        return None, None, None, None, 0, 0, None

    # Length distribution + outlier guard (drop mis-segmented cycles)
    lengths = np.array([r[0].shape[0] for r in raw])
    med = np.median(lengths)
    print(f"lengths: min={lengths.min()} med={med:.0f} max={lengths.max()} n={len(raw)}")

    keep = [r for r in raw if 0.5 * med <= r[0].shape[0] <= 2.0 * med]
    print(f"dropped {len(raw) - len(keep)} outlier cycles")

    # Time-normalize every cycle to TARGET_LEN points
    X_full = np.array([resample_cycle(r[0]) for r in keep])
    y_full = np.array([r[1] for r in keep])
    ids_full = np.array([r[2] for r in keep])

    # Encode labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_full)
    y_encoded = to_categorical(y_int)
    class_names = le.classes_

    num_features = X_full.shape[2]
    num_classes = y_encoded.shape[1]

    # Train/test split by unique file paths (cycle-level split)
    unique_ids = np.unique(ids_full)
    train_ids, test_ids = train_test_split(unique_ids, train_size=TRAIN_RATIO, random_state=42, shuffle=True)
    train_idx = np.where(np.isin(ids_full, train_ids))[0]
    test_idx = np.where(np.isin(ids_full, test_ids))[0]

    X_train, X_test = X_full[train_idx], X_full[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    return X_train, X_test, y_train, y_test, num_features, num_classes, class_names


def build_model(model_type, sequence_length, num_features, num_classes):
    """Build and compile a model of the requested type: 'lstm', 'gru', or 'cnn'."""
    if model_type == 'lstm':
        model = Sequential([
            LSTM(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='LSTM'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'gru':
        model = Sequential([
            GRU(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='GRU'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'cnn':
        model = Sequential([
            # Convolution over time steps
            Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(sequence_length, num_features)),
            Dropout(DROPOUT_RATE),
            Conv1D(filters=64, kernel_size=5, activation='relu'),
            GlobalAveragePooling1D(),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def train_and_compare():
    """Train and evaluate multiple architectures on the same split."""
    X_train, X_test, y_train, y_test, num_features, num_classes, class_names = load_and_split_data()
    if X_train is None or X_train.shape[0] < 1:
        print("Insufficient data; training aborted.")
        return

    sequence_length = X_train.shape[1]
    model_types = ['lstm', 'gru', 'cnn']

    for mtype in model_types:
        print(f"\n=== Training {mtype.upper()} model ===")
        model = build_model(mtype, sequence_length, num_features, num_classes)
        weights_path = f'{WEIGHTS_PREFIX}_{mtype}_IMU_{sequence_length}.weights.h5'

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
            ModelCheckpoint(weights_path, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=0)
        ]

        history = model.fit(
            X_train, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_test, y_test),
            callbacks=callbacks,
            verbose=1
        )

        # Load best weights and evaluate
        try:
            model.load_weights(weights_path)
        except Exception as e:
            print(f"Warning: Could not load best weights for {mtype}: {e}")

        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"{mtype.upper()} Test Loss: {loss:.4f} | Test Accuracy: {accuracy:.4f}")

        # Classification report
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        y_true = np.argmax(y_test, axis=1)
        print(f"\n{mtype.upper()} Classification Report:")
        print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0, digits=4))


if __name__ == '__main__':
    tf.get_logger().setLevel('INFO')
    train_and_compare()

2026-07-27 16:14:24.860834: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785183264.883026 1228536 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785183264.889666 1228536 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1785183264.906535 1228536 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785183264.906557 1228536 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785183264.906559 1228536 computation_placer.cc:177] computation placer alr

lengths: min=81 med=138 max=250 n=5166
dropped 0 outlier cycles

=== Training LSTM model ===


I0000 00:00:1785183299.710924 1228536 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 992 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1785183299.716322 1228536 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 44343 MB memory:  -> device: 1, name: NVIDIA RTX A6000, pci bus id: 0000:25:00.0, compute capability: 8.6
I0000 00:00:1785183299.717855 1228536 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 46412 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1785183299.719492 1228536 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 46412 MB memory:  -> device: 3, name: NVIDIA RTX A6000, pci bus id: 0000:c1:00.0, compute capability: 8.6
/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarni

Epoch 1/25


I0000 00:00:1785183302.256807 1229973 cuda_dnn.cc:529] Loaded cuDNN version 90701


...

130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.3363 - loss: 1.4377 - val_accuracy: 0.6277 - val_loss: 0.9771
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5579 - loss: 1.0528 - val_accuracy: 0.7089 - val_loss: 0.8204
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6491 - loss: 0.8994 - val_accuracy: 0.7660 - val_loss: 0.7033
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7198 - loss: 0.7550 - val_accuracy: 0.8027 - val_loss: 0.5960
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7457 - loss: 0.6762 - val_accuracy: 0.8356 - val_loss: 0.5111
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7901 - loss: 0.5662 - val_accuracy: 0.8598 - val_loss: 0.4465
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8088 - loss: 0.5134 - val_accuracy: 0.8936 - val_loss: 0.3675
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8444 - loss: 0.4435 - val_accuracy: 0.9043 - 

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1785183370.349695 1229971 service.cc:152] XLA service 0x7fec88111fc0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1785183370.349737 1229971 service.cc:160]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1785183370.349742 1229971 service.cc:160]   StreamExecutor device (1): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1785183370.349744 1229971 service.cc:160]   StreamExecutor device (2): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1785183370.349746 1229971 service.cc:160]   StreamExecutor device (3): NVIDIA RTX A6000, Compute C

 77/130 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3705 - loss: 8.4972  

I0000 00:00:1785183372.637212 1229971 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


130/130 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.4323 - loss: 6.1608 - val_accuracy: 0.7176 - val_loss: 0.8811
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7066 - loss: 0.6965 - val_accuracy: 0.8607 - val_loss: 0.5039
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7864 - loss: 0.5471 - val_accuracy: 0.8346 - val_loss: 0.4582
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8248 - loss: 0.4587 - val_accuracy: 0.9207 - val_loss: 0.2861
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8727 - loss: 0.3481 - val_accuracy: 0.9101 - val_loss: 0.2641
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8617 - loss: 0.3488 - val_accuracy: 0.9439 - val_loss: 0.2049
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9105 - loss: 0.2591 - val_accuracy: 0.9671 - val_loss: 0.1614
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9268 - loss: 0.2203 - val_accuracy: 0.9149 - val

In [2]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Conv1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# =========================
# Configuration and constants
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100          # time-normalized points per cycle (101 for exact 0-100%)
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']
FEATURE_COLUMNS = [
    'IMU101_v0', 'IMU101_v1',
    'IMU103_v0', 'IMU103_v1',
    'IMU104_v0', 'IMU104_v1', 'IMU301_v0', 'IMU301_v1',
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6',
    'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]

# Model hyperparameters
UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32

# Checkpoint naming (kept distinct from the old padded-input runs)
WEIGHTS_PREFIX = 'best_timenorm'


def resample_cycle(features, target_len=TARGET_LEN):
    """Time-normalize one cycle onto a fixed number of points (0-100% of cycle)."""
    n = features.shape[0]
    kind = 'cubic' if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_and_split_data():
    """Load cycles, select features, time-normalize, encode labels, and split."""
    raw = []

    if not os.path.exists(ROOT_DIR):
        return None, None, None, None, 0, 0, None

    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                file_path = os.path.join(root, filename)

                # Parse label and cycle id from filename
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cycle_id_str = raw_id_part.split('_')[-1]

                if label not in VALID_LABELS or not cycle_id_str.isdigit():
                    continue

                try:
                    df = pd.read_csv(file_path)
                except Exception:
                    continue

                # Feature selection
                if any(col not in df.columns for col in FEATURE_COLUMNS):
                    continue
                df = df[FEATURE_COLUMNS]
                cycle_features = df.values
                if cycle_features.shape[0] < 2 or cycle_features.shape[1] == 0:
                    continue

                raw.append((cycle_features, label, file_path))

    if not raw:
        return None, None, None, None, 0, 0, None

    # Length distribution + outlier guard (drop mis-segmented cycles)
    lengths = np.array([r[0].shape[0] for r in raw])
    med = np.median(lengths)
    print(f"lengths: min={lengths.min()} med={med:.0f} max={lengths.max()} n={len(raw)}")

    keep = [r for r in raw if 0.5 * med <= r[0].shape[0] <= 2.0 * med]
    print(f"dropped {len(raw) - len(keep)} outlier cycles")

    # Time-normalize every cycle to TARGET_LEN points
    X_full = np.array([resample_cycle(r[0]) for r in keep])
    y_full = np.array([r[1] for r in keep])
    ids_full = np.array([r[2] for r in keep])

    # Encode labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_full)
    y_encoded = to_categorical(y_int)
    class_names = le.classes_

    num_features = X_full.shape[2]
    num_classes = y_encoded.shape[1]

    # Train/test split by unique file paths (cycle-level split)
    unique_ids = np.unique(ids_full)
    train_ids, test_ids = train_test_split(unique_ids, train_size=TRAIN_RATIO, random_state=42, shuffle=True)
    train_idx = np.where(np.isin(ids_full, train_ids))[0]
    test_idx = np.where(np.isin(ids_full, test_ids))[0]

    X_train, X_test = X_full[train_idx], X_full[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    return X_train, X_test, y_train, y_test, num_features, num_classes, class_names


def build_model(model_type, sequence_length, num_features, num_classes):
    """Build and compile a model of the requested type: 'lstm', 'gru', or 'cnn'."""
    if model_type == 'lstm':
        model = Sequential([
            LSTM(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='LSTM'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'gru':
        model = Sequential([
            GRU(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='GRU'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'cnn':
        model = Sequential([
            # Convolution over time steps
            Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(sequence_length, num_features)),
            Dropout(DROPOUT_RATE),
            Conv1D(filters=64, kernel_size=5, activation='relu'),
            GlobalAveragePooling1D(),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def train_and_compare():
    """Train and evaluate multiple architectures on the same split."""
    X_train, X_test, y_train, y_test, num_features, num_classes, class_names = load_and_split_data()
    if X_train is None or X_train.shape[0] < 1:
        print("Insufficient data; training aborted.")
        return

    sequence_length = X_train.shape[1]
    model_types = ['lstm', 'gru', 'cnn']

    for mtype in model_types:
        print(f"\n=== Training {mtype.upper()} model ===")
        model = build_model(mtype, sequence_length, num_features, num_classes)
        weights_path = f'{WEIGHTS_PREFIX}_{mtype}_IMU+Pressure_{sequence_length}.weights.h5'

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
            ModelCheckpoint(weights_path, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=0)
        ]

        history = model.fit(
            X_train, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_test, y_test),
            callbacks=callbacks,
            verbose=1
        )

        # Load best weights and evaluate
        try:
            model.load_weights(weights_path)
        except Exception as e:
            print(f"Warning: Could not load best weights for {mtype}: {e}")

        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"{mtype.upper()} Test Loss: {loss:.4f} | Test Accuracy: {accuracy:.4f}")

        # Classification report
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        y_true = np.argmax(y_test, axis=1)
        print(f"\n{mtype.upper()} Classification Report:")
        print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0, digits=4))


if __name__ == '__main__':
    tf.get_logger().setLevel('INFO')
    train_and_compare()

lengths: min=81 med=138 max=250 n=5166
dropped 0 outlier cycles

=== Training LSTM model ===
Epoch 1/25


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.3810 - loss: 1.3460 - val_accuracy: 0.6838 - val_loss: 0.8844
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6422 - loss: 0.8717 - val_accuracy: 0.8308 - val_loss: 0.6167
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7682 - loss: 0.6182 - val_accuracy: 0.9023 - val_loss: 0.4193
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8546 - loss: 0.4647 - val_accuracy: 0.9217 - val_loss: 0.3239
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8800 - loss: 0.3599 - val_accuracy: 0.9449 - val_loss: 0.2571
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9061 - loss: 0.3043 - val_accuracy: 0.9458 - val_loss: 0.2136
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9255 - loss: 0.2480 - val_accuracy: 0.9671 - val_loss: 0.1532
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9455 - loss: 0.1955 - val_accuracy: 0.9671 - val

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.4491 - loss: 1.3365 - val_accuracy: 0.8462 - val_loss: 0.5790
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7491 - loss: 0.6434 - val_accuracy: 0.9120 - val_loss: 0.3489
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8625 - loss: 0.4062 - val_accuracy: 0.9449 - val_loss: 0.2267
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9150 - loss: 0.2814 - val_accuracy: 0.9526 - val_loss: 0.1751
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9358 - loss: 0.2154 - val_accuracy: 0.9584 - val_loss: 0.1537
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9510 - loss: 0.1688 - val_accuracy: 0.9671 - val_loss: 0.1233
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9644 - loss: 0.1349 - val_accuracy: 0.9613 - val_loss: 0.1194
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9676 - loss: 0.1272 - val_accuracy: 0.9642 - val

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.6492 - loss: 3.1578 - val_accuracy: 0.9217 - val_loss: 0.2579
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8957 - loss: 0.2738 - val_accuracy: 0.9072 - val_loss: 0.2450
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9212 - loss: 0.2066 - val_accuracy: 0.9681 - val_loss: 0.1245
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9457 - loss: 0.1529 - val_accuracy: 0.9265 - val_loss: 0.1874
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9493 - loss: 0.1511 - val_accuracy: 0.9526 - val_loss: 0.1582
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9712 - loss: 0.0932 - val_accuracy: 0.9787 - val_loss: 0.0727
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9756 - loss: 0.0819 - val_accuracy: 0.9729 - val_loss: 0.1037
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9649 - loss: 0.0986 - val_accuracy: 0.9797 - val

In [3]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Conv1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# =========================
# Configuration and constants
# =========================
ROOT_DIR = 'All_10person_Cycles'
TARGET_LEN = 100          # time-normalized points per cycle (101 for exact 0-100%)
INTERP_KIND = 'cubic'     # 'linear' avoids overshoot on sharp pressure transitions
TRAIN_RATIO = 0.8
VALID_LABELS = ['back', 'front', 'normal', 'side']
FEATURE_COLUMNS = [
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6',
    'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15'
]

# Model hyperparameters
UNITS = 64
DROPOUT_RATE = 0.3
EPOCHS = 25
BATCH_SIZE = 32

# Checkpoint naming (kept distinct from the old padded-input runs)
WEIGHTS_PREFIX = 'best_timenorm'


def resample_cycle(features, target_len=TARGET_LEN):
    """Time-normalize one cycle onto a fixed number of points (0-100% of cycle)."""
    n = features.shape[0]
    kind = INTERP_KIND if n >= 4 else 'linear'
    old_t = np.linspace(0, 1, n)
    new_t = np.linspace(0, 1, target_len)
    return interp1d(old_t, features, axis=0, kind=kind)(new_t)


def load_and_split_data():
    """Load cycles, select features, time-normalize, encode labels, and split."""
    raw = []

    if not os.path.exists(ROOT_DIR):
        return None, None, None, None, 0, 0, None

    for root, dirs, files in os.walk(ROOT_DIR):
        if root.endswith('Abnormal'):
            for filename in files:
                if not filename.endswith('.csv'):
                    continue
                file_path = os.path.join(root, filename)

                # Parse label and cycle id from filename
                parts = filename.replace('.csv', '').split('__')
                if len(parts) != 2:
                    continue
                name_label_part, raw_id_part = parts
                label = name_label_part.split('_')[-1]
                if raw_id_part.count('_') > 1:
                    continue
                if raw_id_part.count('_') == 1 and not raw_id_part.startswith('cycle_'):
                    continue
                cycle_id_str = raw_id_part.split('_')[-1]

                if label not in VALID_LABELS or not cycle_id_str.isdigit():
                    continue

                try:
                    df = pd.read_csv(file_path)
                except Exception:
                    continue

                # Feature selection
                if any(col not in df.columns for col in FEATURE_COLUMNS):
                    continue
                df = df[FEATURE_COLUMNS]
                cycle_features = df.values
                if cycle_features.shape[0] < 2 or cycle_features.shape[1] == 0:
                    continue

                raw.append((cycle_features, label, file_path))

    if not raw:
        return None, None, None, None, 0, 0, None

    # Length distribution + outlier guard (drop mis-segmented cycles)
    lengths = np.array([r[0].shape[0] for r in raw])
    med = np.median(lengths)
    print(f"lengths: min={lengths.min()} med={med:.0f} max={lengths.max()} n={len(raw)}")

    keep = [r for r in raw if 0.5 * med <= r[0].shape[0] <= 2.0 * med]
    print(f"dropped {len(raw) - len(keep)} outlier cycles")

    # Time-normalize every cycle to TARGET_LEN points
    X_full = np.array([resample_cycle(r[0]) for r in keep])
    y_full = np.array([r[1] for r in keep])
    ids_full = np.array([r[2] for r in keep])

    # Encode labels
    le = LabelEncoder()
    y_int = le.fit_transform(y_full)
    y_encoded = to_categorical(y_int)
    class_names = le.classes_

    num_features = X_full.shape[2]
    num_classes = y_encoded.shape[1]

    # Train/test split by unique file paths (cycle-level split)
    unique_ids = np.unique(ids_full)
    train_ids, test_ids = train_test_split(unique_ids, train_size=TRAIN_RATIO, random_state=42, shuffle=True)
    train_idx = np.where(np.isin(ids_full, train_ids))[0]
    test_idx = np.where(np.isin(ids_full, test_ids))[0]

    X_train, X_test = X_full[train_idx], X_full[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    return X_train, X_test, y_train, y_test, num_features, num_classes, class_names


def build_model(model_type, sequence_length, num_features, num_classes):
    """Build and compile a model of the requested type: 'lstm', 'gru', or 'cnn'."""
    if model_type == 'lstm':
        model = Sequential([
            LSTM(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='LSTM'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'gru':
        model = Sequential([
            GRU(UNITS, input_shape=(sequence_length, num_features), return_sequences=False, name='GRU'),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    elif model_type == 'cnn':
        model = Sequential([
            # Convolution over time steps
            Conv1D(filters=64, kernel_size=5, activation='relu', input_shape=(sequence_length, num_features)),
            Dropout(DROPOUT_RATE),
            Conv1D(filters=64, kernel_size=5, activation='relu'),
            GlobalAveragePooling1D(),
            Dropout(DROPOUT_RATE),
            Dense(num_classes, activation='softmax')
        ])
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def train_and_compare():
    """Train and evaluate multiple architectures on the same split."""
    X_train, X_test, y_train, y_test, num_features, num_classes, class_names = load_and_split_data()
    if X_train is None or X_train.shape[0] < 1:
        print("Insufficient data; training aborted.")
        return

    sequence_length = X_train.shape[1]
    model_types = ['lstm', 'gru', 'cnn']

    for mtype in model_types:
        print(f"\n=== Training {mtype.upper()} model ===")
        model = build_model(mtype, sequence_length, num_features, num_classes)
        weights_path = f'{WEIGHTS_PREFIX}_{mtype}_pressure_{sequence_length}.weights.h5'

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
            ModelCheckpoint(weights_path, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=0)
        ]

        history = model.fit(
            X_train, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_test, y_test),
            callbacks=callbacks,
            verbose=1
        )

        # Load best weights and evaluate
        try:
            model.load_weights(weights_path)
        except Exception as e:
            print(f"Warning: Could not load best weights for {mtype}: {e}")

        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f"{mtype.upper()} Test Loss: {loss:.4f} | Test Accuracy: {accuracy:.4f}")

        # Classification report
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
        y_true = np.argmax(y_test, axis=1)
        print(f"\n{mtype.upper()} Classification Report:")
        print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0, digits=4))


if __name__ == '__main__':
    tf.get_logger().setLevel('INFO')
    train_and_compare()

lengths: min=81 med=138 max=250 n=5166
dropped 0 outlier cycles

=== Training LSTM model ===
Epoch 1/25


/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.3367 - loss: 1.3594 - val_accuracy: 0.6605 - val_loss: 0.8351
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7005 - loss: 0.7363 - val_accuracy: 0.8356 - val_loss: 0.4616
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8525 - loss: 0.4159 - val_accuracy: 0.8743 - val_loss: 0.3519
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8922 - loss: 0.3140 - val_accuracy: 0.9130 - val_loss: 0.2444
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9313 - loss: 0.2238 - val_accuracy: 0.9217 - val_loss: 0.2275
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9385 - loss: 0.1933 - val_accuracy: 0.9246 - val_loss: 0.2794
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.9535 - loss: 0.1608 - val_accuracy: 0.9400 - val_loss: 0.1830
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9582 - loss: 0.1462 - val_accuracy: 0.9468 - va

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.3143 - loss: 1.4696 - val_accuracy: 0.6983 - val_loss: 0.8159
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7231 - loss: 0.6814 - val_accuracy: 0.8095 - val_loss: 0.4492
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8459 - loss: 0.3947 - val_accuracy: 0.9130 - val_loss: 0.2606
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9253 - loss: 0.2369 - val_accuracy: 0.9275 - val_loss: 0.2213
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9527 - loss: 0.1628 - val_accuracy: 0.9255 - val_loss: 0.2047
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9524 - loss: 0.1515 - val_accuracy: 0.9400 - val_loss: 0.1828
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9584 - loss: 0.1333 - val_accuracy: 0.9458 - val_loss: 0.1689
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9699 - loss: 0.1005 - val_accuracy: 0.9458 - val

/apps/anaconda/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


130/130 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - accuracy: 0.6742 - loss: 2.1761 - val_accuracy: 0.9342 - val_loss: 0.2448
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9024 - loss: 0.2972 - val_accuracy: 0.9487 - val_loss: 0.1981
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9195 - loss: 0.2525 - val_accuracy: 0.9420 - val_loss: 0.1891
Epoch 4/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9389 - loss: 0.2004 - val_accuracy: 0.9555 - val_loss: 0.1646
Epoch 5/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9470 - loss: 0.1637 - val_accuracy: 0.9439 - val_loss: 0.1675
Epoch 6/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9490 - loss: 0.1529 - val_accuracy: 0.9555 - val_loss: 0.1484
Epoch 7/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9552 - loss: 0.1392 - val_accuracy: 0.9603 - val_loss: 0.1395
Epoch 8/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9568 - loss: 0.1266 - val_accuracy: 0.9632 - val